# **Aligning multilingual medieval corpus with Aquilign**

This notebook enables the execution of multilingual medieval alignment on several witnesses (inputs in .txt format).


# 1. Libraries, code, models import

Install first what will be necessary for the execution of the code:

In [ ]:
!pip install numpyencoder==0.3.2
!pip install langid==1.1.6
!pip install faiss-cpu==1.14.2
!pip install sentence_transformers==4.1.0

Clone the repository of the workshop (a fork of Aquilign):

In [ ]:
!git clone https://github.com/ProMeText/multilingual-medieval-aligner-workshop.git

- Download the **model** for the **alignment** ; here, our finetuned model:

In [ ]:
!pip install gdown -q
!gdown --id '1VCJogBP7q840WJ3-UOE5lC9w8D-oauny'
!unzip /content/model_labse_bibles_clean.zip && rm /content/model_labse_bibles_clean.zip

Add path to the system and init.py:


In [ ]:
import sys
sys.path.append('/content/multilingual-medieval-aligner-workshop')

!touch /content/multilingual-medieval-aligner-workshop/aquilign/__init__.py
!touch /content/multilingual-medieval-aligner-workshop/aquilign/align/__init__.py
!touch /content/multilingual-medieval-aligner-workshop/aquilign/preproc/__init__.py


Import the libraries and the functions:

In [ ]:

import string
from numpyencoder import NumpyEncoder
import sys
import numpy as np
import random
import os

import aquilign.align.graph_merge as graph_merge
import aquilign.align.utils as utils
import aquilign.preproc.tok_apply as bert_tokenize
import aquilign.preproc.regex_tokenization as regex_tokenization
from aquilign.align.encoder import Encoder
from aquilign.align.aligner import Bertalign, Bertalign_Embbed

import pandas as pd
import argparse
import glob

# 2. Function for the pairs and main class Aligner


## Function of creation of pairs

In [ ]:
def create_pairs(full_list: list, main_wit_index: int) -> list[tuple]:
    """
    From a list of witnesses and the main witness index, create all possible pairs with this witness. Returns a list
    of tuples with the main wit and the wit to compare it to
    """
    pairs = []
    main_wit = full_list.pop(int(main_wit_index))
    for wit in full_list:
        pairs.append((main_wit, wit))
    return pairs


##  Main class Aligner

In [ ]:
class Aligner:
    """
    Aligner initializes alignment, based on Bertalign
    """

    def __init__(self,
                 model,
                 corpus_limit: None,
                 max_align=3,
                 out_dir="out",
                 use_punctuation=True,
                 input_dir="in",
                 main_wit=None,
                 prefix=None,
                 device="cpu",
                 tokenizer="regexp",
                 tok_models=None,
                 multilingual=True
                 ):
        self.model = model
        self.alignment_dict = dict()
        self.text_dict = dict()
        self.files_path = glob.glob(f"{input_dir}/*/*.txt")
        self.device = device
        self.multilingual_segmentation_model = multilingual
        assert any([main_wit in path for path in
                    self.files_path]), "Main wit doesn't match witnesses paths, please check arguments. " \
                                       f"Main wit: {main_wit}, other wits: {self.files_path}"
        print(self.files_path)
        self.main_file_index = next(index for index, path in enumerate(self.files_path) if main_wit in path)
        self.corpus_limit = corpus_limit
        self.max_align = max_align
        self.out_dir = out_dir
        self.use_punctuation = use_punctuation
        self.prefix = prefix
        self.tokenizer = tokenizer
        self.tok_models = tok_models
        self.wit_pairs = create_pairs(self.files_path, self.main_file_index)

        try:
            os.mkdir(f"result_dir")
        except FileExistsError:
            pass
        try:
            os.mkdir(f"result_dir/{self.out_dir}/")
        except FileExistsError:
            pass

        # Let's check the paths are correct
        for file in self.files_path:
            assert os.path.isfile(file), f"Check the path: {file}"

    def parallel_align(self):
        """
        This function procedes to the alignments two by two and then merges the alignments into a single alignement
        """
        pivot_text = self.wit_pairs[0][0]
        if self.multilingual_segmentation_model and self.tokenizer == "bert-based":
            pivot_text_lang = "ml"
        else:
            pivot_text_lang = pivot_text.split("/")[-2]
        if not self.tokenizer:
            print("Code not implemented for input texts as lists. Exiting")
            exit(0)
        elif self.tokenizer == "regexp":
            first_tokenized_text = utils.clean_tokenized_content(
                regex_tokenization.regex_tokenization(input_file=pivot_text,
                                                      corpus_limit=self.corpus_limit,
                                                      use_punctuation=True,
                                                      lang=pivot_text_lang))
        else:
            first_tokenized_text = bert_tokenize.tokenize_text(input_file=pivot_text,
                                                               corpus_limit=self.corpus_limit,
                                                               remove_punct=False,
                                                               tok_models=self.tok_models,
                                                               output_dir=self.out_dir,
                                                               device=self.device,
                                                               lang=pivot_text_lang)

        sentence_embeddings = Bertalign_Embbed(model=self.model,
                                               max_align=self.max_align,
                                               sents=first_tokenized_text)
        pivot_vecs, pivot_lens, pivot_search_simple_vecs = sentence_embeddings.return_embbeds()

        assert first_tokenized_text != [], "Error with the tokenized text of the base witness"



        main_wit_name = self.wit_pairs[0][0].split("/")[-1].split(".")[0]
        utils.write_json(f"result_dir/{self.out_dir}/tokenized_{main_wit_name}.json", first_tokenized_text)
        utils.write_tokenized_text(f"result_dir/{self.out_dir}/tokenized_{main_wit_name}.txt", first_tokenized_text)

        # Let's loop and align each pair
        # We randomize the pairs. It can help resolving memory issue.
        random.shuffle(self.wit_pairs)
        for index, (main_wit, wit_to_compare) in enumerate(self.wit_pairs):
            main_wit_name = main_wit.split("/")[-1].split(".")[0]
            wit_to_compare_name = wit_to_compare.split("/")[-1].split(".")[0]
            if self.multilingual_segmentation_model:
                current_wit_lang = "ml"
            else:
                current_wit_lang = wit_to_compare.split("/")[-2]
            print(len(first_tokenized_text))
            if self.tokenizer is None:
                pass
            elif self.tokenizer == "regexp":
                second_tokenized_text = utils.clean_tokenized_content(
                    regex_tokenization.regex_tokenization(input_file=wit_to_compare,
                                                          corpus_limit=self.corpus_limit,
                                                          use_punctuation=True,
                                                          lang=current_wit_lang))
            else:
                second_tokenized_text = bert_tokenize.tokenize_text(input_file=wit_to_compare,
                                                                    corpus_limit=self.corpus_limit,
                                                                    remove_punct=False,
                                                                    tok_models=self.tok_models,
                                                                    output_dir=self.out_dir,
                                                                    device=self.device,
                                                                    lang=current_wit_lang)
            assert second_tokenized_text != [], f"Error with the tokenized text of the witnesse to be compared {wit_to_compare_name}"
            utils.write_json(f"result_dir/{self.out_dir}/tokenized_{wit_to_compare_name}.json", second_tokenized_text)
            utils.write_tokenized_text(f"result_dir/{self.out_dir}/tokenized_{wit_to_compare_name}.txt",
                                       second_tokenized_text)

            # This dict will be used to create the alignment table in csv format
            self.text_dict[0] = first_tokenized_text
            self.text_dict[index + 1] = second_tokenized_text

            # Let's align the texts
            print(f"Aligning {main_wit} with {wit_to_compare}")

            # Tests on parameters
            profile = 0
            if profile == 0:
                margin = True
                len_penality = True
            else:
                margin = False
                len_penality = True
            aligner = Bertalign(model=self.model,
                                src_sents=first_tokenized_text,
                                src_lens=pivot_lens,
                                src_vecs=pivot_vecs,
                                search_simple_vecs=pivot_search_simple_vecs,
                                tgt=second_tokenized_text,
                                max_align=self.max_align,
                                win=5,
                                skip=-.2,
                                margin=margin,
                                len_penalty=len_penality,
                                device=self.device)
            aligner.align_sents()

            # We append the result to the alignment dictionnary
            self.alignment_dict[index] = aligner.result
            utils.write_json(f"result_dir/{self.out_dir}/alignment_{str(index)}.json", aligner.result)
            utils.save_alignment_results(aligner.result, first_tokenized_text, second_tokenized_text,
                                         f"{main_wit_name}_{wit_to_compare_name}", self.out_dir)
        utils.write_json(f"result_dir/{self.out_dir}/alignment_dict.json", self.alignment_dict)

    def save_final_result(self, merged_alignments: list, delimiter="\t"):
        """
        Saves result to csv file
        """

        all_wits = [self.wit_pairs[0][0]] + [pair[1] for pair in self.wit_pairs]
        filenames = [wit.split("/")[-1].replace(".txt", "") for wit in all_wits]
        with open(f"result_dir/{self.out_dir}/final_result.csv", "w") as output_text:
            output_text.write(delimiter + delimiter.join(filenames) + "\n")
            translation_table = {letter: index for index, letter in enumerate(string.ascii_lowercase)}
            for alignment_unit in merged_alignments:
                output_text.write("|".join(value for value in alignment_unit['a']) + delimiter)
                for index, witness in enumerate(merged_alignments[0]):
                    output_text.write("|".join(self.text_dict[translation_table[witness]][int(value)] for value in
                                               alignment_unit[witness]))
                    if index + 1 != len(merged_alignments[0]):
                        output_text.write(delimiter)
                output_text.write("\n")

        with open(f"result_dir/{self.out_dir}/readable.csv", "w") as output_text:
            output_text.write(delimiter.join(filenames) + "\n")
            translation_table = {letter: index for index, letter in enumerate(string.ascii_lowercase)}
            for alignment_unit in merged_alignments:
                for index, witness in enumerate(merged_alignments[0]):
                    output_text.write(" ".join(self.text_dict[translation_table[witness]][int(value)] for value in
                                               alignment_unit[witness]))
                    if index + 1 != len(merged_alignments[0]):
                        output_text.write(delimiter)
                output_text.write("\n")

        with open(f"result_dir/{self.out_dir}/final_result_as_index.csv", "w") as output_text:
            output_text.write(delimiter + delimiter.join(filenames) + "\n")
            for alignment_unit in merged_alignments:
                for index, witness in enumerate(merged_alignments[0]):
                    output_text.write("|".join(value for value in
                                               alignment_unit[witness]))
                    if index + 1 != len(merged_alignments[0]):
                        output_text.write(delimiter)
                output_text.write("\n")

        data = pd.read_csv(f"result_dir/{self.out_dir}/final_result.csv", delimiter="\t")
        # Convert the DataFrame to an HTML table
        html_table = data.to_html()
        full_html_file = f"""<html>
                          <head>
                          <title>Alignement final</title>
                            <meta http-equiv="Content-Type" content="text/html; charset=utf-8">
                            </head>
                          <body>
                          {html_table}
                          </body>
                    </html>"""
        with open(f"result_dir/{self.out_dir}/final_result.html", "w") as output_html:
            output_html.write(full_html_file)

# 3. Arguments

Using command line interface, the command is `python3 main.py -o lancelot -i multilingual-medieval-aligner-workshop/data/ii-48_extrait -mw multilingual-medieval-aligner-workshop/data/ii-48_extrait/fr/micha-ii-48.txt -d cpu -t bert-based`.

Here, we have to give the different arguments:

- name of the **output folder**:

In [ ]:
out_dir = "lancelot"

- path to the **input folder**, i.e. the folder which contains the folders which contain the .txt files (each folder should be named with the ISO code language of the language in which the .txt file is written).
*Caveat*: if you want to align your own data, make sure the data dir is organized as follows:
```
lancelot/
├── en
│   └── lancelot.txt
└── lat
    └── lancelot.txt
```
There is no specific rule for the filenames, but each text must be stored in a language dir.

In [ ]:
input_dir = "multilingual-medieval-aligner-workshop/data/to-align/ii-48_extrait"
# check the path
assert input_dir != None,  "Input dir is mandatory"

- path to the witness we choose as **main witness** (or pivot witness):

In [ ]:
main_wit = "multilingual-medieval-aligner-workshop/data/to-align/ii-48_extrait/fr/micha-ii-48.txt"
# check the path
assert main_wit != None,  "Main wit path is mandatory"

**Note** : If you want to test the code on you own data, you can change the values of **input_dir** and
**main_wit**. You can get the data e.g. from your drive (if need help, see the cells 'Input file' of the segmenter-application.ipynb notebook).

- the **device**. It will use the GPU if you have selected the correct runtime type:

In [ ]:
import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"

- the **tokenizer**, i.e. the method of tokenization we choose (bert-based or regex-based, which is turn on by default):


In [ ]:
tokenizer = "bert-based"
# check that a valid value has been passed as argument
assert tokenizer in ["None", "regexp", "bert-based"], "Authorized values for tokenizer are: None, regexp, bert-based"
if tokenizer == "None":
    tokenizer = None

We can add other arguments, and change the default values.

- **prefix** if we want to have a prefix on our output files:


In [ ]:
prefix = ""

- **usage of punctuation** in the segmentation/tokenization phase (default is True):

In [ ]:
use_punctuation = False

In [ ]:
corpus_limit = None
if corpus_limit:
    corpus_limit = float(corpus_limit)


In [ ]:
# Initialize model
models = {0: "/content/model_labse_bibles_clean", 1: "LaBSE"}
model = Encoder(models[int(0)], device=device)

### Segmentation models

We list the **available segmentation** models :


In [ ]:
tok_models = {"fr":
                  {"model": "ProMeText/aquilign_french_segmenter",
                   "tokenizer": "dbmdz/bert-base-french-europeana-cased",
                   "tokens_per_example": 12},
              "es": {"model": "ProMeText/aquilign_spanish_segmenter",
                     "tokenizer": "dccuchile/bert-base-spanish-wwm-cased",
                     "tokens_per_example": 30},
              "it": {"model": "ProMeText/aquilign_italian_segmenter",
                     "tokenizer": "dbmdz/bert-base-italian-xxl-cased",
                     "tokens_per_example": 12},
              "la": {"model": "ProMeText/aquilign_segmenter_latin",
                     "tokenizer": "LuisAVasquez/simple-latin-bert-uncased",
                     "tokens_per_example": 50},
              "ml": {"model": "ProMeText/aquilign-multilingual-segmenter",
                         "tokenizer": "google-bert/bert-base-multilingual-cased",
                         "tokens_per_example": 100}
              }

By default, multilingual is turn to True. Add `multilingual=False` if you want to use models by language. The list of available models can be found *supra*, as `tok_models`, identified with the language each one has been trained on.

# 4. Executing the alignment

## Instanciation

Instanciation of the class with the arguments defined *supra*

In [ ]:
MyAligner = Aligner(model, corpus_limit=corpus_limit,
                    max_align=3,
                    out_dir=out_dir,
                    use_punctuation=use_punctuation,
                    input_dir=input_dir,
                    main_wit=main_wit,
                    prefix=prefix,
                    device=device,
                    tokenizer=tokenizer,
                    tok_models=tok_models)

## Proceed to the alignment

The alignment is executed, pair by pair, after the segmentation of the .txt files.

In [ ]:
MyAligner.parallel_align()

Different files are created:
- `-tok.txt`: tokenized files
- `alignment.json`: alignment by index on each pair
- `XXX-1_XXX2_as_index.tsv`: idem in tsv format
-  `XXX-1_XXX2.csv`: alignment by text on each pair

The `Align` class produces a dictionary that lists all the pairwise alignments. This dictionary is stored in a json file, `alignment_dict.json`:

In [ ]:
utils.write_json(f"result_dir/{out_dir}/alignment_dict.json", MyAligner.alignment_dict)
align_dict = utils.read_json(f"result_dir/{out_dir}/alignment_dict.json")

The next step is to **merge the individual alignment tables** into a **single table**. To do this, we project each alignment unit into a graph (an object comprising nodes linked together by edges). There is a common indicator for all alignments: **simply connect all the nodes together** to **merge the alignment tables**.


In [ ]:
list_of_merged_alignments = graph_merge.merge_alignment_table(MyAligner.alignment_dict)

## Tests

Nodes loss test:

In [ ]:
print("Testing results consistency")
possible_witnesses = string.ascii_lowercase[:len(align_dict) + 1]
tested_table = utils.test_tables_consistency(list_of_merged_alignments, possible_witnesses)

## Final output file production

We save the files and produce the final table as a HTML document:

In [ ]:
# Let's save the final tables (indices and texts)
MyAligner.save_final_result(merged_alignments=list_of_merged_alignments)

We can see the final alignment table at: /content/result_dir/lancelot/final_result.html